In [37]:
from osgeo import gdal
import numpy as np
import pandas as pd
import os

In [38]:
df = pd.read_csv("split_64_perblock.csv")
df.head()

,Unnamed: 0,image_name,image_path,mask_path,split_name,dir_name
0,0,g0_xoff_0_yoff_0,NaN,NaN,TRAIN,data
1,1,g12_xoff_0_yoff_0,NaN,NaN,TRAIN,data
2,2,g13_xoff_0_yoff_0,NaN,NaN,TEST,data
3,3,g14_xoff_0_yoff_0,NaN,NaN,VALID,data
4,4,g14_xoff_0_yoff_64,NaN,NaN,VALID,data


In [39]:
def getBands(img):
	bands = np.empty((img.RasterCount, img.RasterYSize, img.RasterXSize), dtype="float32")
	for i in range(img.RasterCount):
		bands[i, :, :] = img.GetRasterBand(i + 1).ReadAsArray()

	return bands

In [40]:
SPLIT_KEYS = ["TRAIN", "VALID", "TEST"]
CLASS_KEY = ["HEALTHY", "STRESSED"]

In [41]:
data = {s: [np.empty((s2.shape[0], 0)), np.empty((s2.shape[0], 0))] for s in SPLIT_KEYS}

root = "dataset_cropped_64_coniferous"
for i, row in df.iterrows():
    fname = row["image_name"]
    split = row["split_name"]

    s2_ds = gdal.Open(os.path.join(root, fname + "_S2.tif"))
    s2 = getBands(s2_ds)

    mask_ds = gdal.Open(os.path.join(root, fname + "_LABEL.tif"))
    mask = mask_ds.GetRasterBand(1).ReadAsArray().astype("uint8")

    healthy = s2[:, (mask == 0) & (np.all(s2 > 0, axis=0))]
    stressed = s2[:, (mask == 1) & (np.all(s2 > 0, axis=0))]

    data[split][0] = np.concatenate([data[split][0], healthy], axis=1)
    data[split][1] = np.concatenate([data[split][1], stressed], axis=1)


In [42]:
for key in SPLIT_KEYS:
    print(key, "Healthy:", data[key][0].shape, "Stressed:", data[key][1].shape)

for key in SPLIT_KEYS:
    for i in range(2):
        np.save(os.path.join("dataset_cropped_64_npy", f"{key}_{CLASS_KEY[i]}.npy"), data[key][i])

TRAIN Healthy: (12, 183089) Stressed: (12, 19489)
VALID Healthy: (12, 61584) Stressed: (12, 4300)
TEST Healthy: (12, 129076) Stressed: (12, 17220)


In [45]:
np.random.seed = 1551554242

for key in SPLIT_KEYS:
    for i in range(2):
        d = data[key][i]
        if i == 0:
            d = d[:, np.random.choice(list(range(d.shape[1])), data[key][1].shape[1], replace=False)]

        print(key, CLASS_KEY[i], d.shape)
        np.save(os.path.join("dataset_cropped_64_npy", f"{key}_{CLASS_KEY[i]}_even.npy"), d)

TRAIN HEALTHY (12, 19489)
TRAIN STRESSED (12, 19489)
VALID HEALTHY (12, 4300)
VALID STRESSED (12, 4300)
TEST HEALTHY (12, 17220)
TEST STRESSED (12, 17220)
